# Brain Slice Extraction Pipeline
## Overview
This notebook automates the extraction of individual coronal brain sections from a whole-slide DAPI image. 

### Methodology:
1. **File Selection:** A GUI-based path selector to point the script to your `.png` or `.jpg` slide.
2. **Channel Selection:** DAPI signal is isolated from the Blue channel to maximize SNR.
3. **Interactive Segmentation:** We use a Li-thresholding algorithm with a widget-based 'sensitivity' offset to handle dim staining.
4. **Spatial Sorting:** Slices are ordered by their (Y, X) coordinates to maintain anatomical sequence (Top-Left to Bottom-Right).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from skimage import io, filters, measure, morphology, exposure
from ipywidgets import interact, IntSlider, FloatSlider, Text, Button, VBox, Output
from IPython.display import display, clear_output

# Setting up plotting style
plt.rcParams['figure.figsize'] = [12, 8]

# UI Elements
file_input = Text(value='Slide with slices.jpg', description='File Path:', layout={'width': '50%'})
sens_slider = FloatSlider(value=1.0, min=0.1, max=3.0, step=0.05, description='Sensitivity')
area_slider = IntSlider(value=5000, min=500, max=50000, step=500, description='Min Area')
ui = VBox([file_input, sens_slider, area_slider])
out = Output()

In [ ]:
def analyze_slide(image_path, sensitivity, min_slice_size):
    with out:
        clear_output(wait=True)
        if not os.path.exists(image_path):
            print(f"[ERROR] File '{image_path}' not found. Please check the path.")
            return None, None
            
        # Load and Pre-process
        img = io.imread(image_path)
        # Extract Blue channel for DAPI
        blue = img[:, :, 2] if (img.ndim == 3 and img.shape[2] >= 3) else img
        
        # Contrast Stretching
        p2, p98 = np.percentile(blue, (2, 99.8))
        img_rescale = exposure.rescale_intensity(blue, in_range=(p2, p98))
        
        # Thresholding
        base_thresh = filters.threshold_li(img_rescale)
        thresh = base_thresh * sensitivity
        binary = img_rescale > thresh
        
        # Morphological Cleaning
        binary = morphology.remove_small_objects(binary, min_size=min_slice_size)
        binary = morphology.binary_closing(binary, morphology.disk(5))
        
        # Labeling and Sorting (Row-major)
        labels = measure.label(binary)
        regions = measure.regionprops(labels)
        regions = sorted(regions, key=lambda r: (r.centroid[0] // 300, r.centroid[1]))
        
        # Visualization
        fig, ax = plt.subplots(figsize=(12, 8))
        ax.imshow(img)
        ax.set_title(f"Detected {len(regions)} Brain Slices")
        
        for i, region in enumerate(regions):
            minr, minc, maxr, maxc = region.bbox
            rect = mpatches.Rectangle((minc, minr), maxc - minc, maxr - minr,
                                      fill=False, edgecolor='lime', linewidth=2)
            ax.add_patch(rect)
            ax.text(minc, minr-15, f"#{i+1}", color='lime', fontweight='bold', fontsize=12)
        
        plt.axis('off')
        plt.show()
        return regions, img

# Linking UI to the function
def update_plot(change):
    global current_regions, current_img
    current_regions, current_img = analyze_slide(file_input.value, sens_slider.value, area_slider.value)

file_input.observe(update_plot, names='value')
sens_slider.observe(update_plot, names='value')
area_slider.observe(update_plot, names='value')

display(ui, out)
# Initial trigger
update_plot(None)


In [ ]:
def export_and_preview(regions, img, output_folder='extracted_slices'):
    if regions is None:
        print("[ERROR] No regions detected. Adjust sliders in Cell 3 first.")
        return

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"[LOG] Created directory: {output_folder}")

    num_slices = len(regions)
    print(f"[LOG] Exporting {num_slices} slices...")
    
    # Setup for Grid Preview
    cols = 4
    rows = (num_slices // cols) + (1 if num_slices % cols != 0 else 0)
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows*4))
    axes = axes.flatten()

    for i, region in enumerate(regions):
        minr, minc, maxr, maxc = region.bbox
        pad = 40 # Padding for DeepSlice compatibility
        minr, minc = max(0, minr-pad), max(0, minc-pad)
        maxr, maxc = min(img.shape[0], maxr+pad), min(img.shape[1], maxc+pad)
        
        crop = img[minr:maxr, minc:maxc]
        fname = f"slice_{i+1:02d}.png"
        io.imsave(os.path.join(output_folder, fname), crop)
        
        # Verbose Logging
        print(f"      > Saved {fname} | Centroid: ({int(region.centroid[1])}, {int(region.centroid[0])})")
        
        # Grid Display
        axes[i].imshow(crop)
        axes[i].set_title(f"Slice {i+1}")
        axes[i].axis('off')

    # Hide unused axes
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
        
    plt.tight_layout()
    plt.show()
    print(f"\n[SUCCESS] Extraction complete. Files located in: {os.path.abspath(output_folder)}")

# Execute
export_and_preview(current_regions, current_img)